## 1. Importações e Parâmetros de Execução

In [0]:
import time
from datetime import datetime
from pyspark.sql import functions as F

dbutils.widgets.text("catalogo", "workspace")
dbutils.widgets.text("data_referencia_calculo", "2026-05-22")

catalogo = dbutils.widgets.get("catalogo")
data_referencia_calculo = dbutils.widgets.get("data_referencia_calculo").strip()

if data_referencia_calculo:
    DATA_REFERENCIA_COL = F.to_date(F.lit(data_referencia_calculo))
    print(f"Modo: recorte fixo em {data_referencia_calculo}")
else:
    DATA_REFERENCIA_COL = F.current_date()
    print("Modo: recorte diario (current_date)")

spark.sql(f"USE CATALOG {catalogo}")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

execucao_id = datetime.now().strftime("%Y%m%d_%H%M%S")
inicio_total = time.time()

print(f"Catalogo em uso: {catalogo}")
print(f"execucao_id:     {execucao_id}")

## 2. Funções Utilitárias

- `registrar_dq`: append em `bronze.dq_log` com volumetria e duração por etapa.
- `garantir_existencia`: confere existência e não-vacuidade, levanta exceção em falha.
- `otimizar_delta`: `OPTIMIZE` com `ZORDER`.



In [0]:
def registrar_dq(tabela: str, etapa: str, registros: int, duracao: float, camada: str = "gold"):
    log = spark.createDataFrame(
        [(execucao_id, camada, tabela, etapa, registros, float(duracao), datetime.now())],
        ["execucao_id", "camada", "tabela", "etapa", "registros", "duracao_segundos", "timestamp_log"],
    )
    (
        log.write.format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(f"{catalogo}.bronze.dq_log")
    )


def garantir_existencia(tabela_full: str) -> int:
    # Confere existencia e nao-vacuidade da tabela
    if not spark.catalog.tableExists(tabela_full):
        raise Exception(f"Tabela ausente: {tabela_full}")
    qtd = spark.table(tabela_full).count()
    if qtd == 0:
        raise Exception(f"Tabela vazia: {tabela_full}")
    print(f"[OK] {tabela_full}: {qtd:,} registros")
    return qtd


def otimizar_delta(tabela_full: str, zorder_cols: list):
    # OPTIMIZE + ZORDER
    inicio = time.time()
    try:
        cols = ", ".join(zorder_cols)
        spark.sql(f"OPTIMIZE {tabela_full} ZORDER BY ({cols})")
        dur = time.time() - inicio
        registrar_dq(tabela_full.split('.')[-1], "optimize_zorder", 0, dur)
        print(f"[OK] OPTIMIZE {tabela_full} ZORDER BY ({cols}) em {dur:.1f}s")
    except Exception as e:
        print(f"[WARN] OPTIMIZE falhou em {tabela_full}: {e}")

## 3. Regras de Negócio Aplicadas

### Granularidade

Uma linha por pedido válido (A). Uma linha por mês com pedidos (B). Uma linha por SKU em `silver.dim_produtos`, mesmo sem vínculo nas outras tabelas (C).

### Filtro de venda

Receita, ticket médio, ranking de categoria e estado, métricas de venda do produto, clientes únicos e novos: todos restritos a `status = 'Aprovado'`. As contagens explícitas de status em `gold_vendas_kpis` consideram todos os valores.

### Janelas temporais

`qtd_vendida_30d`, `receita_30d`, `qtd_vendida_90d`, `qtd_tickets_30d` usam `date_sub(data_referencia_calculo, N)` como piso.

### Divisão por zero

Retorna `null` em `taxa_problema`, `taxa_conversao`, `pct_recomendam`, `nota_media`, `ticket_medio` e nas três taxas de `gold_vendas_kpis`.

### Tipos das razões

Percentuais em `decimal(5,2)`, escala 0–100. Frações em `decimal(7,4)`, escala 0–1 (`taxa_conversao` pode ultrapassar 1).

### Top do mês

`categoria_mais_vendida`: top 1 por quantidade vendida, desempate por receita, depois alfabético. `estado_maior_receita`: top 1 por receita, desempate alfabético. Categoria ou estado nulos não concorrem.

### Clientes únicos e novos

Distintos com pelo menos um pedido aprovado no mês. "Novo" usa `min(data_pedido) WHERE status = 'Aprovado'` em window global por `id_cliente`.

### Classificação do produto

Avaliação em ordem, primeira condição verdadeira vence:

| Classe | Condição |
|---|---|
| `Encalhado` | `qtd_vendida_90d = 0` e `ativo = true` |
| `Problematico` | `qtd_vendida_total >= 10` e `taxa_problema > 0.07` |
| `Top Vendedor` | `qtd_vendida_30d > 0` e `receita_30d >= percentil 80` |
| `Estavel` | demais |

Percentil 80 via `approxQuantile` sobre produtos com venda nos últimos 30 dias.

### Produtos com flag de qualidade

A Silver preserva produtos com `categoria_invalida`, `ativo_invalido` ou `preco_invalido`. Em A e C, `categoria` nula vira `"Sem categoria"` via `coalesce`. `preco_atual` espelha exatamente o estado da Silver, sem tratamento adicional.

## 4. Leitura das Silvers

Seis tabelas lidas uma única vez. `garantir_existencia` valida cada uma antes da construção começar.

In [0]:
inicio = time.time()

qtd_pedidos     = garantir_existencia(f"{catalogo}.silver.fat_pedidos")
qtd_produtos    = garantir_existencia(f"{catalogo}.silver.dim_produtos")
qtd_clientes    = garantir_existencia(f"{catalogo}.silver.tb_clientes")
qtd_tickets     = garantir_existencia(f"{catalogo}.silver.tb_tickets")
qtd_avaliacoes  = garantir_existencia(f"{catalogo}.silver.tb_avaliacoes")
qtd_clickstream = garantir_existencia(f"{catalogo}.silver.tb_clickstream")

df_pedidos     = spark.table(f"{catalogo}.silver.fat_pedidos")
df_produtos    = spark.table(f"{catalogo}.silver.dim_produtos")
df_clientes    = spark.table(f"{catalogo}.silver.tb_clientes")
df_tickets     = spark.table(f"{catalogo}.silver.tb_tickets")
df_avaliacoes  = spark.table(f"{catalogo}.silver.tb_avaliacoes")
df_clickstream = spark.table(f"{catalogo}.silver.tb_clickstream")

JANELA_30D = F.date_sub(DATA_REFERENCIA_COL, 30)
JANELA_90D = F.date_sub(DATA_REFERENCIA_COL, 90)

registrar_dq("pedidos_produtos_gold", "leitura_silvers", qtd_pedidos, time.time() - inicio)

## 5. View Enriquecida Reaproveitável

`df_pedidos_enriq` é o JOIN de `fat_pedidos` com `dim_produtos` e `tb_clientes`, materializado em `gold._stg_pedidos_enriq` para alimentar Gold A e Gold B sem recomputação. `broadcast` explícito nos dois JOINs.

Gold C não consome a view: agrega por produto sem precisar de cliente.

In [0]:
inicio = time.time()

df_clientes_dim = df_clientes.select(
    F.col("id_cliente"),
    F.concat_ws(" ", F.col("nome"), F.col("sobrenome")).alias("nome_cliente"),
    F.col("estado").alias("estado_cliente"),
)

df_produtos_dim = df_produtos.select(
    F.col("id_produto"),
    F.col("nome_produto"),
    F.coalesce(F.col("categoria"), F.lit("Sem categoria")).alias("categoria_produto"),
)

df_pedidos_enriq = (
    df_pedidos.alias("p")
    .join(F.broadcast(df_clientes_dim), on="id_cliente", how="left")
    .join(F.broadcast(df_produtos_dim), on="id_produto", how="left")
)

# Materializacao em stage Delta: padrao Serverless para evitar recomputacao
stage_pedidos_enriq = f"{catalogo}.gold._stg_pedidos_enriq"
(
    df_pedidos_enriq.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(stage_pedidos_enriq)
)

df_pedidos_enriq = spark.table(stage_pedidos_enriq)
df_pedidos_enriq.createOrReplaceTempView("vw_pedidos_enriq")

qtd_enriq = df_pedidos_enriq.count()
print(f"View enriquecida materializada em {stage_pedidos_enriq}: {qtd_enriq:,} pedidos")

registrar_dq("pedidos_produtos_gold", "view_enriquecida", qtd_enriq, time.time() - inicio)

## 6. Gold A: `gold_pedidos_enriquecidos`

Fato denormalizada, uma linha por pedido. Projeta sobre `df_pedidos_enriq` as 16 colunas do contrato, aplica `coalesce(categoria_produto, "Sem categoria")` e adiciona os derivados temporais `ano`, `mes` e `trimestre`.

### 6.1 Construção

In [0]:
inicio = time.time()

df_gold_pedidos = df_pedidos_enriq.select(
    F.col("id_pedido"),
    F.col("id_cliente"),
    F.col("id_produto"),
    F.col("data_pedido"),
    F.col("quantidade"),
    F.col("valor_unitario"),
    F.col("valor_total"),
    F.col("status"),
    F.col("metodo_pagamento"),
    F.col("nome_cliente"),
    F.col("estado_cliente"),
    F.col("nome_produto"),
    F.coalesce(F.col("categoria_produto"), F.lit("Sem categoria")).alias("categoria_produto"),
    F.year(F.col("data_pedido")).alias("ano"),
    F.month(F.col("data_pedido")).alias("mes"),
    F.quarter(F.col("data_pedido")).alias("trimestre"),
)

qtd_gold_pedidos = df_gold_pedidos.count()
print(f"gold_pedidos_enriquecidos: {qtd_gold_pedidos:,} pedidos")

registrar_dq("gold_pedidos_enriquecidos", "construcao", qtd_gold_pedidos, time.time() - inicio)

### 6.2 Escrita

In [0]:
inicio = time.time()

(
    df_gold_pedidos.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.gold.gold_pedidos_enriquecidos")
)

registrar_dq("gold_pedidos_enriquecidos", "escrita_gold", qtd_gold_pedidos, time.time() - inicio)
print("[OK] gold.gold_pedidos_enriquecidos escrita")

### 6.3 OPTIMIZE + ZORDER

ZORDER pelas quatro colunas mais filtradas: `data_pedido`, `status`, `categoria_produto`, `id_cliente`.

In [0]:
otimizar_delta(
    f"{catalogo}.gold.gold_pedidos_enriquecidos",
    ["data_pedido", "status", "categoria_produto", "id_cliente"],
)

### 6.4 Quality Gate

1. Volumetria e schema (16 colunas).
2. PK `id_pedido` sem nulo, sem duplicata.
3. FKs `id_cliente` e `id_produto` não nulas.
4. Enriquecimento `nome_cliente` e `nome_produto` sem nulo.
5. Reconciliação `count(gold) = count(silver.fat_pedidos)`.
6. `status` e `metodo_pagamento` no domínio.
7. `|valor_unitario × quantidade − valor_total| ≤ 0.05`.
8. `ano`, `mes`, `trimestre` coerentes com `data_pedido`.

In [0]:
falhas_a = []
df_check_a = spark.table(f"{catalogo}.gold.gold_pedidos_enriquecidos")
qtd_a = df_check_a.count()

# 1. Existencia e volumetria
if qtd_a == 0:
    falhas_a.append("gold_pedidos_enriquecidos: tabela vazia")
else:
    print(f"[OK] volumetria: {qtd_a:,} registros")

# 2. Schema completo
colunas_esperadas = {
    "id_pedido", "id_cliente", "id_produto", "data_pedido",
    "quantidade", "valor_unitario", "valor_total",
    "status", "metodo_pagamento",
    "nome_cliente", "estado_cliente", "nome_produto", "categoria_produto",
    "ano", "mes", "trimestre",
}
faltantes = colunas_esperadas - set(df_check_a.columns)
if faltantes:
    falhas_a.append(f"gold_pedidos_enriquecidos: colunas faltando {faltantes}")
else:
    print("[OK] schema completo")

# 3. PK
nulos_pk = df_check_a.filter(F.col("id_pedido").isNull()).count()
dups_pk  = df_check_a.groupBy("id_pedido").count().filter("count > 1").count()
if nulos_pk > 0:
    falhas_a.append(f"gold_pedidos_enriquecidos: {nulos_pk} id_pedido nulos")
if dups_pk > 0:
    falhas_a.append(f"gold_pedidos_enriquecidos: {dups_pk} id_pedido duplicados")
if nulos_pk == 0 and dups_pk == 0:
    print("[OK] PK id_pedido")

# 4. FKs nao nulas
for fk in ["id_cliente", "id_produto"]:
    n = df_check_a.filter(F.col(fk).isNull()).count()
    if n > 0:
        falhas_a.append(f"gold_pedidos_enriquecidos: {n} {fk} nulos")
    else:
        print(f"[OK] FK {fk} sem nulo")

# 4.1. Enriquecimento obrigatório
sem_nome_cliente = df_check_a.filter(F.col("nome_cliente").isNull()).count()
sem_nome_produto = df_check_a.filter(F.col("nome_produto").isNull()).count()
sem_categoria    = df_check_a.filter(F.col("categoria_produto").isNull()).count()

if sem_nome_cliente > 0:
    falhas_a.append(f"gold_pedidos_enriquecidos: {sem_nome_cliente} pedidos sem nome_cliente após join")

if sem_nome_produto > 0:
    falhas_a.append(f"gold_pedidos_enriquecidos: {sem_nome_produto} pedidos sem nome_produto após join")

if sem_categoria > 0:
    falhas_a.append(f"gold_pedidos_enriquecidos: {sem_categoria} pedidos sem categoria_produto após join")

if sem_nome_cliente == 0 and sem_nome_produto == 0 and sem_categoria == 0:
    print("[OK] enriquecimento de cliente/produto sem nulos críticos")

# 5. Reconciliacao
if qtd_a != qtd_pedidos:
    falhas_a.append(f"gold_pedidos_enriquecidos: reconciliacao falhou (gold={qtd_a}, silver={qtd_pedidos})")
else:
    print(f"[OK] reconciliacao silver.fat_pedidos = gold ({qtd_pedidos:,})")

# 6. Dominios enum
STATUS_VALIDOS = ["Aprovado", "Recusado", "Reembolsado", "Processando"]
PAGAMENTOS_VALIDOS = ["PIX", "Cartao", "Boleto"]
n_status = df_check_a.filter(~F.col("status").isin(STATUS_VALIDOS)).count()
n_pag    = df_check_a.filter(~F.col("metodo_pagamento").isin(PAGAMENTOS_VALIDOS)).count()
if n_status > 0:
    falhas_a.append(f"gold_pedidos_enriquecidos: {n_status} status fora do canonico")
else:
    print("[OK] status dentro do canonico")
if n_pag > 0:
    falhas_a.append(f"gold_pedidos_enriquecidos: {n_pag} metodo_pagamento fora do canonico")
else:
    print("[OK] metodo_pagamento dentro do canonico")

# 7. Coerencia financeira
incoerentes = df_check_a.filter(
    F.abs(F.col("valor_unitario") * F.col("quantidade") - F.col("valor_total")) > 0.05
).count()
if incoerentes > 0:
    falhas_a.append(f"gold_pedidos_enriquecidos: {incoerentes} pedidos com incoerencia financeira")
else:
    print("[OK] coerencia financeira preservada")

# 8. Coerencia temporal
incoerencia_temp = df_check_a.filter(
    (F.col("ano")       != F.year(F.col("data_pedido"))) |
    (F.col("mes")       != F.month(F.col("data_pedido"))) |
    (F.col("trimestre") != F.quarter(F.col("data_pedido")))
).count()
if incoerencia_temp > 0:
    falhas_a.append(f"gold_pedidos_enriquecidos: {incoerencia_temp} derivados temporais incoerentes")
else:
    print("[OK] derivados temporais coerentes com data_pedido")

if falhas_a:
    print("\n".join(falhas_a))
    raise Exception(f"Quality gate falhou em gold_pedidos_enriquecidos com {len(falhas_a)} violacoes")

print("\nQuality gate aprovado: gold_pedidos_enriquecidos pronta")

## 7. Gold B: `gold_vendas_kpis`

Agregação mensal em SQL com cinco CTEs sobre `vw_pedidos_enriq`. Uma linha por mês com pedidos. Dezoito colunas no contrato: contagens por status, valores monetários, taxas, contagens de cliente e os dois top 1 do mês.

### 7.1 Construção em SQL/CTEs

- `cte_base`: contagens por status, receita aprovada, `qtd_clientes_unicos`.
- `cte_primeira_compra_aprovada` + `cte_novos_mes`: primeira compra aprovada por cliente.
- `cte_top_categoria`: top 1 do mês por quantidade vendida.
- `cte_top_estado`: top 1 do mês por receita aprovada.
- `NULLIF` em todos os denominadores.

In [0]:
inicio = time.time()

# Adiciona ano/mes ao vw_pedidos_enriq para o SQL
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW vw_pedidos_com_anomes AS
    SELECT
        *,
        YEAR(data_pedido)  AS ano,
        MONTH(data_pedido) AS mes
    FROM vw_pedidos_enriq
""")

df_gold_kpis = spark.sql("""
WITH cte_base AS (
    SELECT
        ano,
        mes,
        COUNT(*)                                                       AS qtd_pedidos,
        SUM(CASE WHEN status = 'Aprovado'    THEN 1 ELSE 0 END)        AS qtd_pedidos_aprovados,
        SUM(CASE WHEN status = 'Recusado'    THEN 1 ELSE 0 END)        AS qtd_pedidos_recusados,
        SUM(CASE WHEN status = 'Reembolsado' THEN 1 ELSE 0 END)        AS qtd_pedidos_reembolsados,
        SUM(CASE WHEN status = 'Processando' THEN 1 ELSE 0 END)        AS qtd_pedidos_processando,
        CAST(
            COALESCE(SUM(CASE WHEN status = 'Aprovado' THEN valor_total END), 0)
            AS DECIMAL(18,2)
        ) AS receita_bruta,
        COUNT(DISTINCT CASE WHEN status = 'Aprovado' THEN id_cliente END) AS qtd_clientes_unicos
    FROM vw_pedidos_com_anomes
    GROUP BY ano, mes
),
cte_primeira_compra_aprovada AS (
    SELECT
        id_cliente,
        YEAR(MIN(data_pedido))  AS ano_primeiro,
        MONTH(MIN(data_pedido)) AS mes_primeiro
    FROM vw_pedidos_com_anomes
    WHERE status = 'Aprovado'
    GROUP BY id_cliente
),
cte_novos_mes AS (
    SELECT
        ano_primeiro AS ano,
        mes_primeiro AS mes,
        COUNT(*) AS qtd_clientes_novos
    FROM cte_primeira_compra_aprovada
    GROUP BY ano_primeiro, mes_primeiro
),
cte_cat_rank AS (
    SELECT
        ano,
        mes,
        categoria_produto,
        SUM(quantidade)  AS qtd_vendida_categoria,
        SUM(valor_total) AS receita_categoria,
        ROW_NUMBER() OVER (
            PARTITION BY ano, mes
            ORDER BY SUM(quantidade) DESC, SUM(valor_total) DESC, categoria_produto ASC
        ) AS rk
    FROM vw_pedidos_com_anomes
    WHERE status = 'Aprovado' AND categoria_produto IS NOT NULL
    GROUP BY ano, mes, categoria_produto
),
cte_top_categoria AS (
    SELECT
        ano,
        mes,
        categoria_produto AS categoria_mais_vendida
    FROM cte_cat_rank
    WHERE rk = 1
),
cte_est_rank AS (
    SELECT
        ano,
        mes,
        estado_cliente,
        SUM(valor_total) AS receita_estado,
        ROW_NUMBER() OVER (
            PARTITION BY ano, mes
            ORDER BY SUM(valor_total) DESC, estado_cliente ASC
        ) AS rk
    FROM vw_pedidos_com_anomes
    WHERE status = 'Aprovado' AND estado_cliente IS NOT NULL
    GROUP BY ano, mes, estado_cliente
),
cte_top_estado AS (
    SELECT
        ano,
        mes,
        estado_cliente AS estado_maior_receita
    FROM cte_est_rank
    WHERE rk = 1
)
SELECT
    b.ano,
    b.mes,
    LPAD(CAST(b.ano AS STRING), 4, '0') || '-' || LPAD(CAST(b.mes AS STRING), 2, '0') AS ano_mes,
    b.qtd_pedidos,
    b.qtd_pedidos_aprovados,
    b.qtd_pedidos_recusados,
    b.qtd_pedidos_reembolsados,
    b.qtd_pedidos_processando,
    b.receita_bruta,
    CAST(b.receita_bruta / NULLIF(b.qtd_pedidos_aprovados, 0) AS DECIMAL(18,2)) AS ticket_medio,
    b.qtd_clientes_unicos,
    COALESCE(n.qtd_clientes_novos, 0) AS qtd_clientes_novos,
    CAST(100.0 * b.qtd_pedidos_aprovados    / NULLIF(b.qtd_pedidos, 0) AS DECIMAL(5,2)) AS taxa_aprovacao,
    CAST(100.0 * b.qtd_pedidos_recusados    / NULLIF(b.qtd_pedidos, 0) AS DECIMAL(5,2)) AS taxa_recusa,
    CAST(100.0 * b.qtd_pedidos_reembolsados / NULLIF(b.qtd_pedidos, 0) AS DECIMAL(5,2)) AS taxa_reembolso,
    tc.categoria_mais_vendida,
    te.estado_maior_receita
FROM cte_base b
LEFT JOIN cte_novos_mes     n  ON b.ano = n.ano  AND b.mes = n.mes
LEFT JOIN cte_top_categoria tc ON b.ano = tc.ano AND b.mes = tc.mes
LEFT JOIN cte_top_estado    te ON b.ano = te.ano AND b.mes = te.mes
ORDER BY b.ano, b.mes
""").withColumn("data_referencia_calculo", DATA_REFERENCIA_COL)

qtd_gold_kpis = df_gold_kpis.count()
print(f"gold_vendas_kpis: {qtd_gold_kpis} meses")

registrar_dq("gold_vendas_kpis", "construcao", qtd_gold_kpis, time.time() - inicio)

### 7.2 Escrita

In [0]:
inicio = time.time()

(
    df_gold_kpis.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.gold.gold_vendas_kpis")
)

registrar_dq("gold_vendas_kpis", "escrita_gold", qtd_gold_kpis, time.time() - inicio)
print("[OK] gold.gold_vendas_kpis escrita")

### 7.3 OPTIMIZE + ZORDER

ZORDER em `ano`, `mes`.

In [0]:
otimizar_delta(f"{catalogo}.gold.gold_vendas_kpis", ["ano", "mes"])

### 7.4 Quality Gate

1. Volumetria e schema (18 colunas).
2. PK composta `(ano, mes)` sem nulo, sem duplicata.
3. Soma dos status = `qtd_pedidos`.
4. Soma das taxas = 100 ± 0.10.
5. `ticket_medio` nulo se e somente se `qtd_pedidos_aprovados = 0`.
6. `sum(qtd_pedidos) = count(silver.fat_pedidos)`.
7. Nenhum mês posterior a `data_referencia_calculo`.

In [0]:
falhas_b = []
df_check_b = spark.table(f"{catalogo}.gold.gold_vendas_kpis")
qtd_b = df_check_b.count()

# 1. Volumetria
if qtd_b == 0:
    falhas_b.append("gold_vendas_kpis: tabela vazia")
else:
    print(f"[OK] volumetria: {qtd_b} meses")

# 2. Schema
colunas_esperadas = {
    "ano", "mes", "ano_mes",
    "qtd_pedidos", "qtd_pedidos_aprovados", "qtd_pedidos_recusados",
    "qtd_pedidos_reembolsados", "qtd_pedidos_processando",
    "receita_bruta", "ticket_medio",
    "qtd_clientes_unicos", "qtd_clientes_novos",
    "taxa_aprovacao", "taxa_recusa", "taxa_reembolso",
    "categoria_mais_vendida", "estado_maior_receita",
    "data_referencia_calculo",
}
faltantes = colunas_esperadas - set(df_check_b.columns)
if faltantes:
    falhas_b.append(f"gold_vendas_kpis: colunas faltando {faltantes}")
else:
    print("[OK] schema completo")

# 3. PK composta (ano, mes)
nulos = df_check_b.filter(F.col("ano").isNull() | F.col("mes").isNull()).count()
dups  = df_check_b.groupBy("ano", "mes").count().filter("count > 1").count()
if nulos > 0:
    falhas_b.append(f"gold_vendas_kpis: {nulos} linhas com ano/mes nulos")
if dups > 0:
    falhas_b.append(f"gold_vendas_kpis: {dups} chaves (ano, mes) duplicadas")
if nulos == 0 and dups == 0:
    print("[OK] PK composta (ano, mes)")

# 4. Soma dos status bate com total
soma_invalida = df_check_b.filter(
    F.col("qtd_pedidos") != (
        F.col("qtd_pedidos_aprovados") + F.col("qtd_pedidos_recusados")
        + F.col("qtd_pedidos_reembolsados") + F.col("qtd_pedidos_processando")
    )
).count()
if soma_invalida > 0:
    falhas_b.append(f"gold_vendas_kpis: {soma_invalida} meses com soma dos status incoerente")
else:
    print("[OK] soma dos status bate com qtd_pedidos")

# 5. Soma das taxas converge a 100 com Processando incluso
taxa_invalida = df_check_b.filter(
    F.abs(
        F.col("taxa_aprovacao") + F.col("taxa_recusa") + F.col("taxa_reembolso")
        + (100.0 * F.col("qtd_pedidos_processando") / F.col("qtd_pedidos"))
        - 100.0
    ) > 0.10
).count()
if taxa_invalida > 0:
    falhas_b.append(f"gold_vendas_kpis: {taxa_invalida} meses com soma de taxas fora de 100")
else:
    print("[OK] soma das taxas em torno de 100")

# 6. ticket_medio coerente
ticket_incoerente = df_check_b.filter(
    ((F.col("qtd_pedidos_aprovados") > 0)  & F.col("ticket_medio").isNull())
    | ((F.col("qtd_pedidos_aprovados") == 0) & F.col("ticket_medio").isNotNull())
).count()
if ticket_incoerente > 0:
    falhas_b.append(f"gold_vendas_kpis: {ticket_incoerente} meses com ticket_medio incoerente")
else:
    print("[OK] ticket_medio coerente com qtd_pedidos_aprovados")

# 7. Reconciliacao
soma_qtd_b = df_check_b.agg(F.sum("qtd_pedidos")).collect()[0][0]
if soma_qtd_b != qtd_pedidos:
    falhas_b.append(f"gold_vendas_kpis: reconciliacao falhou (soma_gold={soma_qtd_b}, silver={qtd_pedidos})")
else:
    print(f"[OK] reconciliacao soma_gold = count_silver ({qtd_pedidos:,})")

# 8. Range temporal
if data_referencia_calculo:
    futuros = df_check_b.filter(
        F.to_date(F.concat_ws("-", F.col("ano"), F.lpad(F.col("mes").cast("string"), 2, "0"), F.lit("01")))
        > DATA_REFERENCIA_COL
    ).count()
    if futuros > 0:
        falhas_b.append(f"gold_vendas_kpis: {futuros} meses posteriores a data_referencia_calculo")
    else:
        print("[OK] range temporal coerente")

if falhas_b:
    print("\n".join(falhas_b))
    raise Exception(f"Quality gate falhou em gold_vendas_kpis com {len(falhas_b)} violacoes")

print("\nQuality gate aprovado: gold_vendas_kpis pronta")

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {stage_pedidos_enriq}")
print(f"[OK] Stage {stage_pedidos_enriq} dropada")

## 8. Gold C: `gold_produto_performance`

Visão analítica por SKU. Quatro agregados independentes em torno de `id_produto` (vendas, tickets, avaliações, clickstream), unidos via LEFT JOIN com `dim_produtos`. `coalesce(metrica, 0)` nas contagens, razões com `null` em denominador zero. Classificação dimensional aplicada por último.

### 8.1 Agregado de Vendas

Filtra `status = 'Aprovado'`. Soma `quantidade` e `valor_total` por produto, total e nas janelas 30d/90d.

In [0]:
inicio = time.time()

df_aprovados = df_pedidos.filter(F.col("status") == "Aprovado")

agg_vendas = df_aprovados.groupBy("id_produto").agg(
    F.sum("quantidade").alias("qtd_vendida_total"),
    F.sum(F.when(F.col("data_pedido") >= JANELA_30D, F.col("quantidade")).otherwise(0)).alias("qtd_vendida_30d"),
    F.sum(F.when(F.col("data_pedido") >= JANELA_90D, F.col("quantidade")).otherwise(0)).alias("qtd_vendida_90d"),
    F.sum("valor_total").cast("decimal(18,2)").alias("receita_total"),
    F.sum(F.when(F.col("data_pedido") >= JANELA_30D, F.col("valor_total")).otherwise(0)).cast("decimal(18,2)").alias("receita_30d"),
)

registrar_dq("gold_produto_performance", "agg_vendas", agg_vendas.count(), time.time() - inicio)

### 8.2 Agregado de Tickets

`silver.tb_tickets` não carrega `id_produto`. Atribuição feita pela ponte `ticket → id_pedido → fat_pedidos → id_produto`. Janela 30d sobre `to_date(data_abertura)`.

In [0]:
inicio = time.time()

# Tickets não possuem id_produto diretamente.
# Para medir tickets por produto, usamos id_pedido como ponte com silver.fat_pedidos.
tickets_com_produto = (
    df_tickets
    .filter(F.col("id_pedido").isNotNull())
    .join(
        df_pedidos.select("id_pedido", "id_produto").dropDuplicates(["id_pedido"]),
        on="id_pedido",
        how="inner"
    )
    .filter(F.col("id_produto").isNotNull())
)

agg_tickets = (
    tickets_com_produto
    .groupBy("id_produto")
    .agg(
        F.count("*").alias("qtd_tickets_associados"),
        F.sum(
            F.when(F.to_date(F.col("data_abertura")) >= JANELA_30D, 1)
             .otherwise(0)
        ).alias("qtd_tickets_30d")
    )
)

registrar_dq(
    "gold_produto_performance",
    "agg_tickets",
    agg_tickets.count(),
    time.time() - inicio
)

### 8.3 Agregado de Avaliações

`qtd_avaliacoes`, `nota_media` (1–5), `pct_recomendam` (0–100).

In [0]:
inicio = time.time()

df_aval_prod = df_avaliacoes.filter(F.col("id_produto").isNotNull())

agg_avaliacoes = df_aval_prod.groupBy("id_produto").agg(
    F.count("*").alias("qtd_avaliacoes"),
    F.avg("nota_produto").cast("decimal(3,2)").alias("nota_media"),
    F.sum(F.when(F.col("recomenda") == True, 1).otherwise(0)).alias("_qtd_recomendam"),
    F.sum(F.when(F.col("recomenda").isNotNull(), 1).otherwise(0)).alias("_qtd_com_resposta"),
).withColumn(
    "pct_recomendam",
    F.when(
        F.col("_qtd_com_resposta") > 0,
        (100.0 * F.col("_qtd_recomendam") / F.col("_qtd_com_resposta")).cast("decimal(5,2)"),
    ).otherwise(F.lit(None).cast("decimal(5,2)")),
).drop("_qtd_recomendam", "_qtd_com_resposta")

registrar_dq("gold_produto_performance", "agg_avaliacoes", agg_avaliacoes.count(), time.time() - inicio)

### 8.4 Agregado de Clickstream

Filtra `id_produto` não nulo. Conta `tipo_evento = 'visualizacao_produto'` em `qtd_visualizacoes` e `tipo_evento = 'adicao_carrinho'` em `qtd_carrinho`.

In [0]:
inicio = time.time()

df_click_prod = df_clickstream.filter(F.col("id_produto").isNotNull())

agg_clickstream = df_click_prod.groupBy("id_produto").agg(
    F.sum(F.when(F.col("tipo_evento") == "visualizacao_produto", 1).otherwise(0)).alias("qtd_visualizacoes"),
    F.sum(F.when(F.col("tipo_evento") == "adicao_carrinho",      1).otherwise(0)).alias("qtd_carrinho"),
)

registrar_dq("gold_produto_performance", "agg_clickstream", agg_clickstream.count(), time.time() - inicio)

### 8.5 Composição Final

LEFT JOIN dos quatro agregados sobre `dim_produtos`. Produto sem vínculo entra com zero nas contagens e `null` nas razões. `taxa_problema = qtd_tickets / qtd_vendida_total`. `taxa_conversao = qtd_vendida / qtd_visualizacoes`. Ambas `null` quando o denominador é zero.

In [0]:
inicio = time.time()

df_perf_base = (
    df_produtos.select(
        "id_produto",
        "nome_produto",
        F.coalesce(F.col("categoria"), F.lit("Sem categoria")).alias("categoria"),
        F.col("preco").alias("preco_atual"),
        "ativo",
    )
    .join(F.broadcast(agg_vendas),      on="id_produto", how="left")
    .join(F.broadcast(agg_tickets),     on="id_produto", how="left")
    .join(F.broadcast(agg_avaliacoes),  on="id_produto", how="left")
    .join(F.broadcast(agg_clickstream), on="id_produto", how="left")
)

df_perf_base = (
    df_perf_base
    .withColumn("qtd_vendida_total",      F.coalesce(F.col("qtd_vendida_total"),      F.lit(0)))
    .withColumn("qtd_vendida_30d",        F.coalesce(F.col("qtd_vendida_30d"),        F.lit(0)))
    .withColumn("qtd_vendida_90d",        F.coalesce(F.col("qtd_vendida_90d"),        F.lit(0)))
    .withColumn("receita_total",          F.coalesce(F.col("receita_total"),          F.lit(0).cast("decimal(18,2)")))
    .withColumn("receita_30d",            F.coalesce(F.col("receita_30d"),            F.lit(0).cast("decimal(18,2)")))
    .withColumn("qtd_tickets_associados", F.coalesce(F.col("qtd_tickets_associados"), F.lit(0)))
    .withColumn("qtd_tickets_30d",        F.coalesce(F.col("qtd_tickets_30d"),        F.lit(0)))
    .withColumn("qtd_avaliacoes",         F.coalesce(F.col("qtd_avaliacoes"),         F.lit(0)))
    .withColumn("qtd_visualizacoes",      F.coalesce(F.col("qtd_visualizacoes"),      F.lit(0)))
    .withColumn("qtd_carrinho",           F.coalesce(F.col("qtd_carrinho"),           F.lit(0)))
)

df_perf_base = (
    df_perf_base
    .withColumn("taxa_problema",
        F.when(F.col("qtd_vendida_total") > 0,
               (F.col("qtd_tickets_associados") / F.col("qtd_vendida_total")).cast("decimal(7,4)"))
         .otherwise(F.lit(None).cast("decimal(7,4)"))
    )
    .withColumn("taxa_conversao",
        F.when(F.col("qtd_visualizacoes") > 0,
               (F.col("qtd_vendida_total") / F.col("qtd_visualizacoes")).cast("decimal(7,4)"))
         .otherwise(F.lit(None).cast("decimal(7,4)"))
    )
)

registrar_dq("gold_produto_performance", "composicao_base", df_perf_base.count(), time.time() - inicio)

### 8.6 Classificação

`approxQuantile(receita_30d, 0.80)` calculado sobre o subconjunto com `qtd_vendida_30d > 0`. Limiar de `taxa_problema` fixado em `0.07`. Avaliação em ordem: Encalhado, Problematico, Top Vendedor, Estavel. Primeira condição verdadeira vence.

In [0]:
inicio = time.time()

LIMIAR_TAXA_PROBLEMA = 0.07

produtos_com_venda_30d = df_perf_base.filter(F.col("qtd_vendida_30d") > 0)

if produtos_com_venda_30d.count() > 0:
    perc80 = produtos_com_venda_30d.approxQuantile("receita_30d", [0.80], 0.01)[0]
    print(f"Percentil 80 de receita_30d: {perc80}")
    perc80_expr = F.lit(float(perc80)).cast("decimal(18,2)")
else:
    perc80 = None
    perc80_expr = F.lit(None).cast("decimal(18,2)")
    print("Nenhum produto com venda nos ultimos 30d, ninguem se classifica como Top Vendedor")

df_perf = df_perf_base.withColumn(
    "classificacao",
    F.when((F.col("qtd_vendida_90d") == 0) & (F.col("ativo") == True),
           F.lit("Encalhado"))
     .when(
         (F.col("qtd_vendida_total") >= 10) &
         (F.col("taxa_problema") > F.lit(LIMIAR_TAXA_PROBLEMA)),
         F.lit("Problematico")
     )
     .when(
         (F.col("qtd_vendida_30d") > 0) &
         (perc80_expr.isNotNull()) &
         (F.col("receita_30d") >= perc80_expr),
         F.lit("Top Vendedor")
     )
     .otherwise(F.lit("Estavel")),
)

registrar_dq("gold_produto_performance", "classificacao", df_perf.count(), time.time() - inicio)

### 8.7 Escrita

In [0]:
inicio = time.time()

df_gold_perf = df_perf.select(
    "id_produto", "nome_produto", "categoria",
    "preco_atual", "ativo",
    "qtd_vendida_total", "qtd_vendida_30d", "qtd_vendida_90d",
    "receita_total", "receita_30d",
    "qtd_tickets_associados", "qtd_tickets_30d", "taxa_problema",
    "qtd_avaliacoes", "nota_media", "pct_recomendam",
    "qtd_visualizacoes", "qtd_carrinho", "taxa_conversao",
    "classificacao",
).withColumn("data_referencia_calculo", DATA_REFERENCIA_COL)

(
    df_gold_perf.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.gold.gold_produto_performance")
)

qtd_gold_perf = spark.table(f"{catalogo}.gold.gold_produto_performance").count()
print(f"gold_produto_performance: {qtd_gold_perf} produtos")

registrar_dq("gold_produto_performance", "escrita_gold", qtd_gold_perf, time.time() - inicio)

### 8.8 OPTIMIZE + ZORDER

ZORDER em `id_produto` (PK), `classificacao` e `categoria`.

In [0]:
otimizar_delta(
    f"{catalogo}.gold.gold_produto_performance",
    ["id_produto", "classificacao", "categoria"],
)

### 8.9 Quality Gate

1. Volumetria e schema (21 colunas).
2. PK `id_produto` sem nulo, sem duplicata.
3. `count(gold) = count(silver.dim_produtos)`.
4. `classificacao` em `{Encalhado, Problematico, Top Vendedor, Estavel}`.
5. `qtd_vendida_30d ≤ qtd_vendida_90d ≤ qtd_vendida_total`.
6. `qtd_tickets_30d ≤ qtd_tickets_associados`.
7. `receita_30d ≤ receita_total`.
8. Razões não nulas só com denominador positivo.
9. `nota_media` em `[1, 5]`. `pct_recomendam` em `[0, 100]`.
10. `taxa_problema` e `taxa_conversao` ≥ 0.

In [0]:
falhas_c = []
df_check_c = spark.table(f"{catalogo}.gold.gold_produto_performance")
qtd_c = df_check_c.count()

# 1. Volumetria
if qtd_c == 0:
    falhas_c.append("gold_produto_performance: tabela vazia")
else:
    print(f"[OK] volumetria: {qtd_c} produtos")

# 2. Schema
colunas_esperadas = {
    "id_produto", "nome_produto", "categoria", "preco_atual", "ativo",
    "qtd_vendida_total", "qtd_vendida_30d", "qtd_vendida_90d",
    "receita_total", "receita_30d",
    "qtd_tickets_associados", "qtd_tickets_30d", "taxa_problema",
    "qtd_avaliacoes", "nota_media", "pct_recomendam",
    "qtd_visualizacoes", "qtd_carrinho", "taxa_conversao",
    "classificacao", "data_referencia_calculo",
}
faltantes = colunas_esperadas - set(df_check_c.columns)
if faltantes:
    falhas_c.append(f"gold_produto_performance: colunas faltando {faltantes}")
else:
    print("[OK] schema completo")

# 3. PK
nulos_pk = df_check_c.filter(F.col("id_produto").isNull()).count()
dups_pk  = df_check_c.groupBy("id_produto").count().filter("count > 1").count()
if nulos_pk > 0:
    falhas_c.append(f"gold_produto_performance: {nulos_pk} id_produto nulos")
if dups_pk > 0:
    falhas_c.append(f"gold_produto_performance: {dups_pk} id_produto duplicados")
if nulos_pk == 0 and dups_pk == 0:
    print("[OK] PK id_produto")

# 4. Reconciliacao com dim_produtos
if qtd_c != qtd_produtos:
    falhas_c.append(f"gold_produto_performance: reconciliacao falhou (gold={qtd_c}, silver={qtd_produtos})")
else:
    print(f"[OK] reconciliacao silver.dim_produtos = gold ({qtd_produtos})")

# 5. Dominio de classificacao
CLASSIFICACOES_VALIDAS = ["Encalhado", "Problematico", "Top Vendedor", "Estavel"]
fora_dominio = df_check_c.filter(~F.col("classificacao").isin(CLASSIFICACOES_VALIDAS)).count()
if fora_dominio > 0:
    falhas_c.append(f"gold_produto_performance: {fora_dominio} classificacoes fora do dominio")
else:
    print("[OK] classificacao dentro do dominio")

# 6. Monotonia qtd
qtd_invertida = df_check_c.filter(
    (F.col("qtd_vendida_30d") > F.col("qtd_vendida_90d"))
    | (F.col("qtd_vendida_90d") > F.col("qtd_vendida_total"))
).count()
if qtd_invertida > 0:
    falhas_c.append(f"gold_produto_performance: {qtd_invertida} produtos com inversao de janela")
else:
    print("[OK] monotonia das janelas de quantidade")

# 6.1. Monotonia dos tickets
tickets_invertidos = df_check_c.filter(
    F.col("qtd_tickets_30d") > F.col("qtd_tickets_associados")
).count()
if tickets_invertidos > 0:
    falhas_c.append(f"gold_produto_performance: {tickets_invertidos} produtos com qtd_tickets_30d > qtd_tickets_associados")
else:
    print("[OK] monotonia das janelas de tickets")

# 7. Monotonia receita
receita_invertida = df_check_c.filter(F.col("receita_30d") > F.col("receita_total")).count()
if receita_invertida > 0:
    falhas_c.append(f"gold_produto_performance: {receita_invertida} produtos com receita_30d > receita_total")
else:
    print("[OK] monotonia das janelas de receita")

# 8. Coerencia das razoes
razao_tp = df_check_c.filter(F.col("taxa_problema").isNotNull()  & (F.col("qtd_vendida_total") == 0)).count()
razao_tc = df_check_c.filter(F.col("taxa_conversao").isNotNull() & (F.col("qtd_visualizacoes") == 0)).count()
razao_pr = df_check_c.filter(F.col("pct_recomendam").isNotNull() & (F.col("qtd_avaliacoes") == 0)).count()
if razao_tp > 0:
    falhas_c.append(f"gold_produto_performance: {razao_tp} taxa_problema com denominador zero")
if razao_tc > 0:
    falhas_c.append(f"gold_produto_performance: {razao_tc} taxa_conversao com denominador zero")
if razao_pr > 0:
    falhas_c.append(f"gold_produto_performance: {razao_pr} pct_recomendam com denominador zero")
if razao_tp == razao_tc == razao_pr == 0:
    print("[OK] razoes coerentes com denominadores")

# 9. Range das notas
nota_fora = df_check_c.filter(
    F.col("nota_media").isNotNull()
    & ((F.col("nota_media") < 1.0) | (F.col("nota_media") > 5.0))
).count()
if nota_fora > 0:
    falhas_c.append(f"gold_produto_performance: {nota_fora} produtos com nota_media fora de [1, 5]")
else:
    print("[OK] nota_media dentro de [1, 5] quando nao nula")

# 9.1. Range de pct_recomendam
pct_fora = df_check_c.filter(
    F.col("pct_recomendam").isNotNull()
    & ((F.col("pct_recomendam") < 0) | (F.col("pct_recomendam") > 100))
).count()
if pct_fora > 0:
    falhas_c.append(f"gold_produto_performance: {pct_fora} produtos com pct_recomendam fora de [0, 100]")
else:
    print("[OK] pct_recomendam dentro de [0, 100] quando não nula")

# 9.2. Range de taxa_problema e taxa_conversao 
taxa_negativa = df_check_c.filter(
    ((F.col("taxa_problema").isNotNull())  & (F.col("taxa_problema")  < 0))
    | ((F.col("taxa_conversao").isNotNull()) & (F.col("taxa_conversao") < 0))
).count()
if taxa_negativa > 0:
    falhas_c.append(f"gold_produto_performance: {taxa_negativa} produtos com taxa negativa")
else:
    print("[OK] taxa_problema e taxa_conversao não-negativas")
    
if falhas_c:
    print("\n".join(falhas_c))
    raise Exception(f"Quality gate falhou em gold_produto_performance com {len(falhas_c)} violacoes")

print("\nQuality gate aprovado: gold_produto_performance pronta")

## 9. Evidências

Distribuições por Gold (9.1, 9.2, 9.3) e validação cruzada das três (9.4).

### 9.1 `gold_pedidos_enriquecidos`

In [0]:
df_a = spark.table(f"{catalogo}.gold.gold_pedidos_enriquecidos")

print("Distribuicao de status")
display(df_a.groupBy("status").count().orderBy(F.desc("count")))

print("Distribuicao de metodo de pagamento")
display(df_a.groupBy("metodo_pagamento").count().orderBy(F.desc("count")))

print("Top 10 categorias por receita aprovada")
display(
    df_a.filter(F.col("status") == "Aprovado")
        .groupBy("categoria_produto")
        .agg(F.count("*").alias("qtd"), F.sum("valor_total").alias("receita"))
        .orderBy(F.desc("receita")).limit(10)
)

print("Top 10 estados por receita aprovada")
display(
    df_a.filter(F.col("status") == "Aprovado")
        .groupBy("estado_cliente")
        .agg(F.count("*").alias("qtd"), F.sum("valor_total").alias("receita"))
        .orderBy(F.desc("receita")).limit(10)
)

### 9.2 `gold_vendas_kpis`

In [0]:
df_b = spark.table(f"{catalogo}.gold.gold_vendas_kpis")

print("Ultimos 12 meses")
display(df_b.orderBy(F.desc("ano"), F.desc("mes")).limit(12))

print("Frequencia das categorias campeas")
display(
    df_b.groupBy("categoria_mais_vendida")
        .agg(F.count("*").alias("meses_no_topo"))
        .orderBy(F.desc("meses_no_topo"))
)

print("Frequencia dos estados campeoes")
display(
    df_b.groupBy("estado_maior_receita")
        .agg(F.count("*").alias("meses_no_topo"))
        .orderBy(F.desc("meses_no_topo"))
)

print("Evolucao anual")
display(
    df_b.groupBy("ano").agg(
        F.sum("receita_bruta").alias("receita_total"),
        F.sum("qtd_pedidos_aprovados").alias("qtd_aprovados"),
        F.sum("qtd_clientes_novos").alias("clientes_novos_ano"),
    ).orderBy("ano")
)

### 9.3 `gold_produto_performance`

In [0]:
df_c = spark.table(f"{catalogo}.gold.gold_produto_performance")

print("Distribuicao da classificacao")
display(
    df_c.groupBy("classificacao").agg(
        F.count("*").alias("qtd_produtos"),
        F.sum("receita_total").alias("receita_consolidada"),
        F.sum("qtd_vendida_total").alias("qtd_vendida_consolidada"),
    ).orderBy(F.desc("qtd_produtos"))
)

print("Top 10 vendedores por receita_30d")
display(
    df_c.filter(F.col("classificacao") == "Top Vendedor")
        .orderBy(F.desc("receita_30d"))
        .select("id_produto", "nome_produto", "categoria",
                "qtd_vendida_30d", "receita_30d", "nota_media", "taxa_problema")
        .limit(10)
)

print("Top 10 problematicos por taxa_problema")
display(
    df_c.filter(F.col("classificacao") == "Problematico")
        .orderBy(F.desc("taxa_problema"))
        .select("id_produto", "nome_produto", "qtd_vendida_total",
                "qtd_tickets_associados", "taxa_problema", "nota_media")
        .limit(10)
)

print("Encalhados ativos com maior visualizacao (candidatos a promocao/descontinuacao)")
display(
    df_c.filter(F.col("classificacao") == "Encalhado")
        .orderBy(F.desc("qtd_visualizacoes"))
        .select("id_produto", "nome_produto", "categoria",
                "preco_atual", "qtd_visualizacoes", "qtd_carrinho")
        .limit(10)
)

print("Cobertura das fontes (% produtos com vinculo em cada fonte)")
display(
    df_c.agg(
        F.count("*").alias("qtd_produtos"),
        F.sum(F.when(F.col("qtd_vendida_total")      > 0, 1).otherwise(0)).alias("com_venda"),
        F.sum(F.when(F.col("qtd_tickets_associados") > 0, 1).otherwise(0)).alias("com_ticket"),
        F.sum(F.when(F.col("qtd_avaliacoes")          > 0, 1).otherwise(0)).alias("com_avaliacao"),
        F.sum(F.when(F.col("qtd_visualizacoes")       > 0, 1).otherwise(0)).alias("com_visualizacao"),
    )
)

### 9.4 Coerência Cruzada

Três invariantes entre as três tabelas:

- `sum(valor_total WHERE status='Aprovado')` em A = `sum(receita_bruta)` em B = `sum(receita_total)` em C.
- `sum(quantidade WHERE status='Aprovado')` em A = `sum(qtd_vendida_total)` em C.
- `count(*)` por status em A = `sum(qtd_pedidos_<status>)` em B.

In [0]:
print("Receita total cruzada (deve ser identica nas tres)")

receita_a = (
    df_a.filter(F.col("status") == "Aprovado")
        .agg(F.sum("valor_total").alias("receita_pedidos_enriq"))
)

receita_b = df_b.agg(F.sum("receita_bruta").alias("receita_vendas_kpis"))

receita_c = df_c.agg(F.sum("receita_total").alias("receita_produto_perf"))

display(receita_a.crossJoin(receita_b).crossJoin(receita_c))

print("\nQuantidade vendida total cruzada (deve ser identica entre A e C)")

qtd_a_aprovados = df_a.filter(F.col("status") == "Aprovado").agg(F.sum("quantidade").alias("qtd_pedidos_enriq"))
qtd_c_total     = df_c.agg(F.sum("qtd_vendida_total").alias("qtd_produto_perf"))

display(qtd_a_aprovados.crossJoin(qtd_c_total))

print("\nDistribuicao de status cruzada (A vs B)")

status_a = (
    df_a.groupBy("status").agg(F.count("*").alias("qtd_A"))
)

status_b = spark.createDataFrame([
    ("Aprovado",     df_b.agg(F.sum("qtd_pedidos_aprovados")).collect()[0][0]    or 0),
    ("Recusado",     df_b.agg(F.sum("qtd_pedidos_recusados")).collect()[0][0]    or 0),
    ("Reembolsado",  df_b.agg(F.sum("qtd_pedidos_reembolsados")).collect()[0][0] or 0),
    ("Processando",  df_b.agg(F.sum("qtd_pedidos_processando")).collect()[0][0]  or 0),
], ["status", "qtd_B"])

display(status_a.join(status_b, on="status", how="outer").orderBy(F.desc("qtd_A")))